In [2]:
%pip install sqlalchemy pandas

Note: you may need to restart the kernel to use updated packages.


In [3]:
import sqlite3
import pandas as pd
from pathlib import Path
from IPython.display import display

In [5]:
# Create a separate DataFrame for SQL
df=pd.read_csv('data/raw/Telco-Customer-Churn.csv')
sql_df = df.copy()

# Convert numerical columns to proper types
sql_df["tenure"] = pd.to_numeric(
    sql_df["tenure"], errors="coerce"
)

sql_df["MonthlyCharges"] = pd.to_numeric(
    sql_df["MonthlyCharges"], errors="coerce"
)

sql_df["TotalCharges"] = pd.to_numeric(
    sql_df["TotalCharges"], errors="coerce"
)

# Ensure churn is consistently encoded
sql_df["ChurnFlag"] = (
    sql_df["Churn"].astype(str).str.strip().map(
        {"Yes": 1, "No": 0}
    )
)

# Handle missing TotalCharges for customers with zero tenure
mask = (
    sql_df["TotalCharges"].isna()
    & sql_df["tenure"].eq(0)
)

sql_df.loc[mask, "TotalCharges"] = 0

# Review the data
print("Rows and columns:", sql_df.shape)
print("Missing values:")
print(sql_df[["tenure", "MonthlyCharges",
              "TotalCharges", "ChurnFlag"]].isna().sum())

display(sql_df.head())

Rows and columns: (7043, 22)
Missing values:
tenure            0
MonthlyCharges    0
TotalCharges      0
ChurnFlag         0
dtype: int64


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,ChurnFlag
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,0
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,No,No,No,One year,No,Mailed check,56.95,1889.50,No,0
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,0
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1


#### Create the dataset and table

In [6]:
Path("data").mkdir(exist_ok=True)

conn = sqlite3.connect("data/customer_intelligence.db")

sql_df.to_sql(
    "customers",
    conn,
    if_exists="replace",
    index=False
)

print("Database created and customers table loaded.")

Database created and customers table loaded.


### Testing the database

In [8]:
query= """ select * from customers
LIMIT 5;
"""
result=pd.read_sql_query(query,conn)
display(result)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,ChurnFlag
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,0
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,No,No,No,One year,No,Mailed check,56.95,1889.50,No,0
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,0
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1


### Count the rows

In [13]:
query=""" Select count(*) as rows from customers;"""
result=pd.read_sql_query(query,conn)
display(result)

,rows
0,7043


#### How many customers churned and stayed?

In [14]:
query = """
SELECT
    Churn,
    COUNT(*) AS customer_count
FROM customers
GROUP BY Churn
ORDER BY customer_count DESC;
"""

display(pd.read_sql_query(query, conn))

,Churn,customer_count
0,No,5174
1,Yes,1869


#### Calculate the overall churn rate

In [15]:
query = """
SELECT
    COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        / NULLIF(COUNT(*), 0),
        2
    ) AS churn_rate_pct
FROM customers;
"""

display(pd.read_sql_query(query, conn))

,total_customers,churned_customers,churn_rate_pct
0,7043,1869,26.54


####  Which contract type has the highest churn rate?

In [16]:
query = """
SELECT
    Contract,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        / NULLIF(COUNT(*), 0),
        2
    ) AS churn_rate_pct
FROM customers
GROUP BY Contract
ORDER BY churn_rate_pct DESC;
"""

display(pd.read_sql_query(query, conn))

,Contract,total_customers,churned_customers,churn_rate_pct
0,Month-to-month,3875,1655,42.71
1,One year,1473,166,11.27
2,Two year,1695,48,2.83


### Which payment methos has the highest churn rate?

In [19]:
query = """
SELECT
    PaymentMethod,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        / NULLIF(COUNT(*), 0),
        2
    ) AS churn_rate_pct
FROM customers
GROUP BY PaymentMethod
ORDER BY churn_rate_pct DESC;
"""

display(pd.read_sql_query(query, conn))

,PaymentMethod,total_customers,churned_customers,churn_rate_pct
0,Electronic check,2365,1071,45.29
1,Mailed check,1612,308,19.11
2,Bank transfer (automatic),1544,258,16.71
3,Credit card (automatic),1522,232,15.24


### Compare average monthly charges for chuned and reatined customers

In [20]:
query = """
SELECT
    Churn,
    COUNT(*) AS customer_count,
    ROUND(AVG(MonthlyCharges), 2) AS avg_monthly_charges,
    ROUND(AVG(TotalCharges), 2) AS avg_total_charges,
    ROUND(AVG(tenure), 1) AS avg_tenure_months
FROM customers
GROUP BY Churn;
"""

display(pd.read_sql_query(query, conn))

,Churn,customer_count,avg_monthly_charges,avg_total_charges,avg_tenure_months
0,No,5174,61.27,2549.91,37.6
1,Yes,1869,74.44,1531.80,18.0


### Find churm rate by tenure group

In [21]:
query = """
WITH tenure_groups AS (
    SELECT
        CASE
            WHEN tenure <= 12 THEN '0-12 months'
            WHEN tenure <= 24 THEN '13-24 months'
            WHEN tenure <= 48 THEN '25-48 months'
            ELSE '49+ months'
        END AS tenure_group,
        tenure,
        Churn
    FROM customers
)
SELECT
    tenure_group,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        / NULLIF(COUNT(*), 0),
        2
    ) AS churn_rate_pct
FROM tenure_groups
GROUP BY tenure_group
ORDER BY
    CASE tenure_group
        WHEN '0-12 months' THEN 1
        WHEN '13-24 months' THEN 2
        WHEN '25-48 months' THEN 3
        ELSE 4
    END;
"""

display(pd.read_sql_query(query, conn))

,tenure_group,total_customers,churned_customers,churn_rate_pct
0,0-12 months,2186,1037,47.44
1,13-24 months,1024,294,28.71
2,25-48 months,1594,325,20.39
3,49+ months,2239,213,9.51


### Find high-value customers who churned

In [22]:
query = """
SELECT
    customerID,
    tenure,
    Contract,
    InternetService,
    MonthlyCharges,
    TotalCharges,
    Churn
FROM customers
WHERE Churn = 'Yes'
  AND MonthlyCharges >= 80
ORDER BY MonthlyCharges DESC, TotalCharges DESC
LIMIT 20;
"""

display(pd.read_sql_query(query, conn))

,customerID,tenure,Contract,InternetService,MonthlyCharges,TotalCharges,Churn
0,8199-ZLLSA,67,One year,Fiber optic,118.35,7804.15,Yes
1,2889-FPWRM,72,One year,Fiber optic,117.80,8684.80,Yes
2,2302-ANTDP,48,Month-to-month,Fiber optic,117.45,5438.90,Yes
3,9053-JZFKV,67,Two year,Fiber optic,116.20,7752.30,Yes
4,1444-VVSGW,70,One year,Fiber optic,115.65,7968.85,Yes
5,0201-OAMXR,70,One year,Fiber optic,115.55,8127.60,Yes
6,4361-BKAXE,41,Month-to-month,Fiber optic,114.50,4527.45,Yes
7,1555-DJEQW,70,Two year,Fiber optic,114.20,7723.90,Yes
8,9158-VCTQB,41,Month-to-month,Fiber optic,113.60,4594.95,Yes
9,7279-BUYWN,41,Month-to-month,Fiber optic,113.20,4689.50,Yes


#### Find segments with above-average churn

In [23]:
query = """
WITH overall AS (
    SELECT
        AVG(CASE WHEN Churn = 'Yes' THEN 1.0 ELSE 0.0 END)
            AS overall_churn_rate
    FROM customers
),
contract_stats AS (
    SELECT
        Contract,
        COUNT(*) AS total_customers,
        SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
            AS churned_customers,
        AVG(CASE WHEN Churn = 'Yes' THEN 1.0 ELSE 0.0 END)
            AS segment_churn_rate
    FROM customers
    GROUP BY Contract
)
SELECT
    cs.Contract,
    cs.total_customers,
    cs.churned_customers,
    ROUND(100.0 * cs.segment_churn_rate, 2) AS segment_churn_pct,
    ROUND(100.0 * o.overall_churn_rate, 2) AS overall_churn_pct
FROM contract_stats cs
CROSS JOIN overall o
WHERE cs.segment_churn_rate > o.overall_churn_rate
ORDER BY cs.segment_churn_rate DESC;
"""

display(pd.read_sql_query(query, conn))

,Contract,total_customers,churned_customers,segment_churn_pct,overall_churn_pct
0,Month-to-month,3875,1655,42.71,26.54


#### What percentage of customers beling to each contract type?

In [24]:
query = """
SELECT
    Contract,
    COUNT(*) AS customer_count,
    ROUND(
        100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
        2
    ) AS customer_share_pct
FROM customers
GROUP BY Contract
ORDER BY customer_share_pct DESC;
"""

display(pd.read_sql_query(query, conn))

,Contract,customer_count,customer_share_pct
0,Month-to-month,3875,55.02
1,Two year,1695,24.07
2,One year,1473,20.91


### Rank contract types by churn rate

In [25]:
query = """
WITH contract_stats AS (
    SELECT
        Contract,
        COUNT(*) AS total_customers,
        AVG(CASE WHEN Churn = 'Yes' THEN 1.0 ELSE 0.0 END)
            AS churn_rate
    FROM customers
    GROUP BY Contract
)
SELECT
    Contract,
    total_customers,
    ROUND(100.0 * churn_rate, 2) AS churn_rate_pct,
    RANK() OVER (ORDER BY churn_rate DESC) AS churn_rank
FROM contract_stats
ORDER BY churn_rank;
"""

display(pd.read_sql_query(query, conn))

,Contract,total_customers,churn_rate_pct,churn_rank
0,Month-to-month,3875,42.71,1
1,One year,1473,11.27,2
2,Two year,1695,2.83,3


#### Calculate cumulative customers by tenure group

In [26]:
query = """
WITH tenure_counts AS (
    SELECT
        CASE
            WHEN tenure <= 12 THEN '0-12 months'
            WHEN tenure <= 24 THEN '13-24 months'
            WHEN tenure <= 48 THEN '25-48 months'
            ELSE '49+ months'
        END AS tenure_group,
        COUNT(*) AS customer_count,
        CASE
            WHEN tenure <= 12 THEN 1
            WHEN tenure <= 24 THEN 2
            WHEN tenure <= 48 THEN 3
            ELSE 4
        END AS group_order
    FROM customers
    GROUP BY tenure_group, group_order
)
SELECT
    tenure_group,
    customer_count,
    SUM(customer_count) OVER (
        ORDER BY group_order
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS cumulative_customers
FROM tenure_counts
ORDER BY group_order;
"""

display(pd.read_sql_query(query, conn))

,tenure_group,customer_count,cumulative_customers
0,0-12 months,2186,2186
1,13-24 months,1024,3210
2,25-48 months,1594,4804
3,49+ months,2239,7043


### Find duplicate cutomers IDs

In [27]:
query = """
SELECT
    customerID,
    COUNT(*) AS record_count
FROM customers
GROUP BY customerID
HAVING COUNT(*) > 1
ORDER BY record_count DESC;
"""

display(pd.read_sql_query(query, conn))

,customerID,record_count


### Identify customers with high monthy charges and short tenure

In [29]:
query = """
SELECT
    customerID,
    tenure,
    MonthlyCharges,
    Contract,
    PaymentMethod,
    OnlineSecurity,
    TechSupport
FROM customers
WHERE tenure <= 12
  AND MonthlyCharges >= 80
ORDER BY MonthlyCharges DESC;
"""

display(pd.read_sql_query(query, conn))

,customerID,tenure,MonthlyCharges,Contract,PaymentMethod,OnlineSecurity,TechSupport
0,1583-IHQZE,12,112.95,Month-to-month,Mailed check,No,Yes
1,3292-PBZEJ,11,111.40,Month-to-month,Electronic check,No,Yes
2,9851-KIELU,10,110.10,Month-to-month,Electronic check,Yes,Yes
3,3992-YWPKO,6,109.90,Month-to-month,Credit card (automatic),No,Yes
4,5760-IFJOZ,3,107.95,Month-to-month,Mailed check,No,No
...,...,...,...,...,...,...,...
472,8821-XNHVZ,1,80.05,Month-to-month,Mailed check,No,No
473,6496-JDSSB,8,80.00,Month-to-month,Bank transfer (automatic),No,No
474,1448-PWKYE,1,80.00,Month-to-month,Electronic check,No,No
475,6941-PMGEP,5,80.00,Month-to-month,Electronic check,No,No


In [ ]:
### Find the top five segments by churnes